# LeetCode 1769 — Minimum Number of Operations to Move All Balls to Each Box bằng Python

Link đề: https://leetcode.com/problems/minimum-number-of-operations-to-move-all-balls-to-each-box/ · Độ khó: **Medium** · Chủ đề: Array, String, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def minOperations(self, boxes: str) -> List[int]:
        ...
```

## 1. Mục tiêu học

1. Hiểu rằng chi phí dồn mọi bóng về hộp `i` là `Σ |i − j|` trên các hộp `j` có bóng (các bóng di chuyển độc lập).
2. Từ vét cạn `O(n²)`, rút ra cách **cập nhật dần** chi phí khi dịch hộp đích sang phải một bước.
3. Cài bằng **hai lượt quét** (trái → phải, phải → trái) với "số bóng bên trái/phải" và "chi phí tích lũy".
4. Cài bằng **tổng tiền tố của vị trí**: `ans[i] = i·cL − sL + sR − i·cR`.
5. Đối chiếu các cách trên chuỗi nhị phân ngẫu nhiên có seed cố định.

## 2. Đề bài & ví dụ

Có `n` hộp, cho chuỗi nhị phân `boxes` độ dài `n`: `boxes[i] == '0'` là hộp rỗng, `'1'` là hộp có **một** quả bóng. Mỗi thao tác: chuyển **một** quả bóng sang hộp **kề** (`|i − j| = 1`). Sau khi chuyển, một hộp có thể chứa nhiều bóng.

Trả về mảng `answer` độ dài `n`, `answer[i]` = số thao tác **tối thiểu** để dồn **mọi** bóng về hộp `i`. Mỗi `answer[i]` tính **độc lập** từ trạng thái ban đầu.

**Ví dụ 1:** `boxes = "110"` → `[1,1,3]`.
- Hộp 0: chuyển bóng ở hộp 1 sang: 1 thao tác.
- Hộp 1: chuyển bóng ở hộp 0 sang: 1 thao tác.
- Hộp 2: bóng hộp 0 đi 2 bước, bóng hộp 1 đi 1 bước: 3 thao tác.

**Ví dụ 2:** `boxes = "001011"` → `[11,8,5,4,3,4]`.

**Ràng buộc:** `1 ≤ n ≤ 2000`, `boxes[i] ∈ {'0','1'}`.

In [1]:
import random
import time
from itertools import accumulate
from typing import List

examples = [("110", [1, 1, 3]), ("001011", [11, 8, 5, 4, 3, 4])]
for b, exp in examples:
    print(f"boxes={b!r} -> kỳ vọng {exp}")

boxes='110' -> kỳ vọng [1, 1, 3]
boxes='001011' -> kỳ vọng [11, 8, 5, 4, 3, 4]


## 3. Kiến thức nền

### 3.1 Lấy vị trí các ký tự `'1'`

`[j for j, c in enumerate(boxes) if c == '1']`.

In [2]:
print([j for j, c in enumerate("001011") if c == "1"])

[2, 4, 5]


### 3.2 Khoảng cách trên đường thẳng `|i − j|`

Một quả bóng ở hộp `j` cần đúng `|i − j|` thao tác để tới hộp `i` (mỗi thao tác đi một bước, không đi tắt được).

### 3.3 Tổng tiền tố với `itertools.accumulate`

`prefix[k] = a[0] + … + a[k−1]` ⇒ tổng đoạn `a[l..r−1] = prefix[r] − prefix[l]`.

In [3]:
a = [0, 0, 1, 0, 1, 1]
prefix = [0] + list(accumulate(a))
print("prefix:", prefix, "| tổng a[2..4] =", prefix[5] - prefix[2])

prefix: [0, 0, 0, 1, 1, 2, 3] | tổng a[2..4] = 2


### 3.4 Duyệt hai chiều

Nhiều bài "mỗi vị trí phụ thuộc vào cả bên trái và bên phải" giải bằng một lượt trái → phải tính phần bên trái, một lượt phải → trái tính phần bên phải, rồi cộng lại.

## 4. Ý tưởng nền tảng / trực giác

Các bóng không cản trở nhau ⇒ `answer[i] = Σ_{j có bóng} |i − j|`.

Tính tay **Ví dụ 2** `"001011"` (bóng ở 2, 4, 5):

| i | |i−2| | |i−4| | |i−5| | tổng |
|---|---|---|---|---|
| 0 | 2 | 4 | 5 | 11 |
| 1 | 1 | 3 | 4 | 8 |
| 2 | 0 | 2 | 3 | 5 |
| 3 | 1 | 1 | 2 | 4 |
| 4 | 2 | 0 | 1 | 3 |
| 5 | 3 | 1 | 0 | 4 |

**Quan sát dịch chuyển:** từ `i` sang `i + 1`, mỗi bóng ở **bên trái hoặc tại** `i` xa thêm 1, mỗi bóng ở **bên phải** `i` gần lại 1. Vậy `answer[i+1] = answer[i] + (#bóng ≤ i) − (#bóng > i)`.

In [4]:
b = "001011"; balls = [j for j, c in enumerate(b) if c == "1"]
ans0 = sum(balls)
cur = ans0; seq = [cur]
for i in range(len(b) - 1):
    left = sum(1 for j in balls if j <= i); right = len(balls) - left
    cur += left - right; seq.append(cur)
print("cập nhật dần:", seq)

cập nhật dần: [11, 8, 5, 4, 3, 4]


## 5. Cách 1 — Brute Force / baseline: cộng `|i − j|` cho mọi cặp

In [5]:
def ops_bruteforce(boxes):
    n = len(boxes)
    return [sum(abs(i - j) for j in range(n) if boxes[j] == "1") for i in range(n)]

for b, exp in examples:
    got = ops_bruteforce(b); print(got, "kỳ vọng", exp); assert got == exp

[1, 1, 3] kỳ vọng [1, 1, 3]
[11, 8, 5, 4, 3, 4] kỳ vọng [11, 8, 5, 4, 3, 4]


### Cách 1'' — Mô phỏng **từng thao tác** (để tin rằng `Σ |i − j|` thật sự đạt được)

Ta mô phỏng đúng luật chơi: lặp, chọn một bóng chưa ở hộp đích, dịch nó **một bước** về phía đích, đếm thao tác. In trạng thái số bóng trong các hộp sau mỗi thao tác cho ví dụ nhỏ.

In [6]:
def simulate(boxes, target, verbose=False):
    state = [int(c) for c in boxes]; ops = 0
    while sum(state) != state[target]:
        j = next(k for k in range(len(state)) if k != target and state[k] > 0)
        step = 1 if target > j else -1
        state[j] -= 1; state[j + step] += 1; ops += 1
        if verbose:
            print(f"  thao tác {ops}: chuyển 1 bóng {j} -> {j + step} | trạng thái {state}")
    return ops

print("Dồn '110' về hộp 2:")
assert simulate("110", 2, verbose=True) == 3
rng = random.Random(0)
for _ in range(200):
    b = "".join(rng.choice("01") for _ in range(rng.randint(1, 10)))
    assert [simulate(b, t) for t in range(len(b))] == ops_bruteforce(b)
print("Mô phỏng khớp công thức Σ|i-j| trên 200 chuỗi ngẫu nhiên")

Dồn '110' về hộp 2:
  thao tác 1: chuyển 1 bóng 0 -> 1 | trạng thái [0, 2, 0]
  thao tác 2: chuyển 1 bóng 1 -> 2 | trạng thái [0, 1, 1]
  thao tác 3: chuyển 1 bóng 1 -> 2 | trạng thái [0, 0, 2]
Mô phỏng khớp công thức Σ|i-j| trên 200 chuỗi ngẫu nhiên


**Độ phức tạp:** `O(n²)` = `4·10⁶` với `n = 2000` — **vẫn qua** giới hạn đề. Nhưng đây là bài luyện kỹ thuật `O(n)` (follow-up tự nhiên khi `n = 10⁵`).

**Nút thắt:** các tổng cho `i` và `i + 1` gần như giống nhau — tính lại từ đầu là lãng phí.

## 6. Ý tưởng thuật toán / bước đột phá

Tách `answer[i] = L[i] + R[i]`:
- `L[i]` = chi phí dồn các bóng ở **bên trái** `i` về `i`;
- `R[i]` = chi phí dồn các bóng ở **bên phải** `i` về `i`.

**Quy tắc cập nhật trái → phải:** gọi `cnt` = số bóng ở `[0, i−1]`. Khi đi từ `i−1` sang `i`, mọi bóng trong đó phải đi thêm 1 bước: `L[i] = L[i−1] + cnt` (trong đó `cnt` đã tính cả `boxes[i−1]`). Tương tự phải → trái cho `R`.

**Invariant (lượt trái):** sau khi xử lý `i`, `cnt` = số bóng trong `boxes[0..i]`, `L[i]` = `Σ_{j<i, có bóng} (i − j)`.

**Cách công thức:** `Σ |i − j| = Σ_{j<i}(i − j) + Σ_{j>i}(j − i) = i·cL − sL + sR − i·cR`, với `cL, sL` = số bóng và tổng vị trí bóng bên trái; `cR, sR` bên phải — lấy từ tổng tiền tố.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Hai mảng `left`, `right`

In [7]:
def ops_two_arrays(boxes):
    n = len(boxes)
    left, right = [0] * n, [0] * n
    cnt = 0
    for i in range(1, n):
        cnt += boxes[i - 1] == "1"
        left[i] = left[i - 1] + cnt
    cnt = 0
    for i in range(n - 2, -1, -1):
        cnt += boxes[i + 1] == "1"
        right[i] = right[i + 1] + cnt
    return [a + b for a, b in zip(left, right)]

for b, _ in examples:
    print(b, ops_two_arrays(b))

110 [1, 1, 3]
001011 [11, 8, 5, 4, 3, 4]


**Code nộp LeetCode (Cách 2):** đổi tên thành `minOperations`. **Độ phức tạp:** Time `O(n)`, Space `O(n)`.

### 7.2 Cách 3 — Hai lượt, cộng thẳng vào `ans` (bớt một mảng)

In [8]:
def ops_two_pass(boxes):
    n = len(boxes); ans = [0] * n
    cnt = cost = 0
    for i in range(n):
        ans[i] += cost
        cnt += boxes[i] == "1"
        cost += cnt                       # chi phí cho vị trí i+1
    cnt = cost = 0
    for i in range(n - 1, -1, -1):
        ans[i] += cost
        cnt += boxes[i] == "1"
        cost += cnt
    return ans

print([ops_two_pass(b) for b, _ in examples])

[[1, 1, 3], [11, 8, 5, 4, 3, 4]]


**Code nộp LeetCode (Cách 3):** xem mục 9. **Độ phức tạp:** Time `O(n)`, Space `O(1)` ngoài kết quả.

### 7.3 Cách 4 — Công thức tổng tiền tố của vị trí

In [9]:
def ops_prefix_formula(boxes):
    n = len(boxes)
    c = [0] * (n + 1); s = [0] * (n + 1)          # số bóng / tổng vị trí trong boxes[0..k-1]
    for k, ch in enumerate(boxes):
        c[k + 1] = c[k] + (ch == "1")
        s[k + 1] = s[k] + (k if ch == "1" else 0)
    ans = []
    for i in range(n):
        cL, sL = c[i], s[i]
        cR, sR = c[n] - c[i + 1], s[n] - s[i + 1]
        ans.append(i * cL - sL + sR - i * cR)
    return ans

print([ops_prefix_formula(b) for b, _ in examples])

[[1, 1, 3], [11, 8, 5, 4, 3, 4]]


**Độ phức tạp Cách 4:** Time `O(n)`, Space `O(n)`. Ưu điểm: trả lời **truy vấn** "chi phí dồn về `i`" cho `i` bất kỳ trong `O(1)` sau tiền xử lý.

### 7.4 Cách 5 — Cập nhật dần một lượt (công thức dịch chuyển ở mục 4)

`ans[0] = Σ vị trí bóng`; `ans[i+1] = ans[i] + (#bóng ≤ i) − (#bóng > i)`.

In [10]:
def ops_shift(boxes):
    n = len(boxes); total = boxes.count("1")
    cur = sum(j for j, ch in enumerate(boxes) if ch == "1")
    ans = [cur]; left = 0
    for i in range(n - 1):
        left += boxes[i] == "1"
        cur += left - (total - left)
        ans.append(cur)
    return ans

print([ops_shift(b) for b, _ in examples])

[[1, 1, 3], [11, 8, 5, 4, 3, 4]]


**Độ phức tạp Cách 5:** Time `O(n)`, Space `O(1)` ngoài kết quả.

## 8. TRACE từng bước

Trace Cách 3 trên **Ví dụ 2** `"001011"`:

| i | (lượt trái) ans += cost | cnt sau | cost sau |
|---|---|---|---|
| 0 | +0 | 0 | 0 |
| 1 | +0 | 0 | 0 |
| 2 | +0 | 1 | 1 |
| 3 | +1 | 1 | 2 |
| 4 | +2 | 2 | 4 |
| 5 | +4 | 3 | 7 |

In [11]:
def trace_two_pass(boxes):
    n = len(boxes); ans = [0] * n
    cnt = cost = 0
    print("Lượt trái -> phải:")
    for i in range(n):
        ans[i] += cost
        cnt += boxes[i] == "1"; cost += cnt
        print(f"  i={i} boxes={boxes[i]} | cộng chi phí trái -> ans={ans} | bóng đã qua={cnt}, chi phí cho i+1={cost}")
    cnt = cost = 0
    print("Lượt phải -> trái:")
    for i in range(n - 1, -1, -1):
        ans[i] += cost
        cnt += boxes[i] == "1"; cost += cnt
        print(f"  i={i} boxes={boxes[i]} | cộng chi phí phải -> ans={ans} | bóng đã qua={cnt}, chi phí cho i-1={cost}")
    return ans

assert trace_two_pass("001011") == [11, 8, 5, 4, 3, 4]

Lượt trái -> phải:
  i=0 boxes=0 | cộng chi phí trái -> ans=[0, 0, 0, 0, 0, 0] | bóng đã qua=0, chi phí cho i+1=0
  i=1 boxes=0 | cộng chi phí trái -> ans=[0, 0, 0, 0, 0, 0] | bóng đã qua=0, chi phí cho i+1=0
  i=2 boxes=1 | cộng chi phí trái -> ans=[0, 0, 0, 0, 0, 0] | bóng đã qua=1, chi phí cho i+1=1
  i=3 boxes=0 | cộng chi phí trái -> ans=[0, 0, 0, 1, 0, 0] | bóng đã qua=1, chi phí cho i+1=2
  i=4 boxes=1 | cộng chi phí trái -> ans=[0, 0, 0, 1, 2, 0] | bóng đã qua=2, chi phí cho i+1=4
  i=5 boxes=1 | cộng chi phí trái -> ans=[0, 0, 0, 1, 2, 4] | bóng đã qua=3, chi phí cho i+1=7
Lượt phải -> trái:
  i=5 boxes=1 | cộng chi phí phải -> ans=[0, 0, 0, 1, 2, 4] | bóng đã qua=1, chi phí cho i-1=1
  i=4 boxes=1 | cộng chi phí phải -> ans=[0, 0, 0, 1, 3, 4] | bóng đã qua=2, chi phí cho i-1=3
  i=3 boxes=0 | cộng chi phí phải -> ans=[0, 0, 0, 4, 3, 4] | bóng đã qua=2, chi phí cho i-1=5
  i=2 boxes=1 | cộng chi phí phải -> ans=[0, 0, 5, 4, 3, 4] | bóng đã qua=3, chi phí cho i-1=8
  i=1 boxes=

## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 3** — hai lượt, `O(n)` thời gian, `O(1)` bộ nhớ phụ.

In [12]:
class Solution:
    def minOperations(self, boxes: str) -> List[int]:
        n = len(boxes)
        ans = [0] * n
        cnt = cost = 0
        for i in range(n):
            ans[i] += cost
            if boxes[i] == "1":
                cnt += 1
            cost += cnt
        cnt = cost = 0
        for i in range(n - 1, -1, -1):
            ans[i] += cost
            if boxes[i] == "1":
                cnt += 1
            cost += cnt
        return ans

for b, exp in examples:
    got = Solution().minOperations(b); print(got, exp); assert got == exp

[1, 1, 3] [1, 1, 3]
[11, 8, 5, 4, 3, 4] [11, 8, 5, 4, 3, 4]


## 10. Vì sao thuật toán đúng

**Bổ đề 1.** `answer[i] = Σ_{j: boxes[j]='1'} |i − j|`. *Cận dưới:* bóng ở `j` cần ít nhất `|i − j|` thao tác (mỗi thao tác dịch một bóng một bước). *Đạt được:* di chuyển từng bóng thẳng về `i`. Các thao tác của các bóng khác nhau tách biệt nên tổng là đúng.

**Lượt trái của Cách 3 (invariant):** ngay **trước** khi xử lý `i`, `cnt` = số bóng ở `[0, i−1]` và `cost = Σ_{j<i, bóng} (i − j)`.
- `i = 0`: `cnt = cost = 0` ✓.
- Từ `i` sang `i+1`: sau khi cộng `boxes[i]`, `cnt` = số bóng ở `[0, i]`; mỗi bóng đó xa `i+1` thêm 1 so với xa `i` (bóng tại `i` từ 0 thành 1) ⇒ `cost_mới = cost + cnt` ✓.

Lượt phải đối xứng. Cộng hai phần ⇒ `Σ_{j<i} + Σ_{j>i} = Σ_j |i − j|` (bóng tại `j = i` đóng góp 0). ∎

## 11. Phân tích Time / Space Complexity

Gọi `n = len(boxes)`.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Vét cạn | `O(n²)` | `O(1)` |
| 2. Hai mảng | `O(n)` | `O(n)` |
| 3. Hai lượt | `O(n)` | `O(1)` |
| 4. Công thức tiền tố | `O(n)` | `O(n)` |
| 5. Cập nhật dần | `O(n)` | `O(1)` |

In [13]:
rng = random.Random(1769)
b = "".join(rng.choice("01") for _ in range(2000))
for name, f in [("brute", ops_bruteforce), ("two_arrays", ops_two_arrays), ("submit", Solution().minOperations),
                ("prefix", ops_prefix_formula), ("shift", ops_shift)]:
    t = time.perf_counter(); r = f(b); print(f"{name:<11} {(time.perf_counter()-t)*1000:7.1f} ms | ans[0]={r[0]}")

brute         157.5 ms | ans[0]=1025217
two_arrays      0.4 ms | ans[0]=1025217
submit          0.2 ms | ans[0]=1025217
prefix          0.4 ms | ans[0]=1025217
shift           0.2 ms | ans[0]=1025217


## 12. Bộ test

In [14]:
ALL = {"two_arrays": ops_two_arrays, "two_pass": ops_two_pass, "prefix": ops_prefix_formula, "shift": ops_shift,
       "submit": Solution().minOperations}
def run_all(b):
    ref = ops_bruteforce(b)
    for k, f in ALL.items():
        assert f(b) == ref, (k, b)
    return ref

for b, exp in examples:
    assert run_all(b) == exp
edge = [("0", [0]), ("1", [0]), ("00", [0, 0]), ("11", [1, 1]), ("101", [2, 2, 2]), ("1111", [6, 4, 4, 6])]
for b, exp in edge:
    assert run_all(b) == exp, b
rng = random.Random(17690)
for _ in range(800):
    p = rng.choice([0.1, 0.5, 0.9])
    run_all("".join("1" if rng.random() < p else "0" for _ in range(rng.randint(1, 40))))
run_all("1" * 2000); run_all("0" * 1999 + "1")
print("Ví dụ, edge case, 800 test ngẫu nhiên, test biên n=2000: OK")

Ví dụ, edge case, 800 test ngẫu nhiên, test biên n=2000: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cộng bóng của hộp hiện tại vào `cnt` **trước** khi cộng `cost` vào `ans`

In [15]:
def wrong_order(boxes):
    n = len(boxes); ans = [0] * n
    cnt = cost = 0
    for i in range(n):
        cnt += boxes[i] == "1"; cost += cnt              # SAI: cập nhật trước
        ans[i] += cost
    cnt = cost = 0
    for i in range(n - 1, -1, -1):
        cnt += boxes[i] == "1"; cost += cnt
        ans[i] += cost
    return ans

b, exp = examples[0]
print("sai :", wrong_order(b), "| đúng:", exp)

sai : [4, 4, 5] | đúng: [1, 1, 3]


**Vì sao sai:** bóng tại chính hộp `i` bị tính 1 bước (và chi phí bị dịch lệch một vị trí). **Sửa:** `ans[i] += cost` **trước**, rồi mới cập nhật `cnt`, `cost`.

### Lỗi 2 — Chỉ tính một phía

In [16]:
def wrong_one_side(boxes):
    n = len(boxes); ans = [0] * n; cnt = cost = 0
    for i in range(n):
        ans[i] += cost; cnt += boxes[i] == "1"; cost += cnt
    return ans

b, exp = examples[1]
print("sai :", wrong_one_side(b), "| đúng:", exp)

sai : [0, 0, 0, 1, 2, 4] | đúng: [11, 8, 5, 4, 3, 4]


**Vì sao sai:** bóng ở bên phải hộp `i` cũng phải di chuyển về. **Sửa:** thêm lượt phải → trái.

### Lỗi 3 — Nghĩ rằng chỉ cần đếm số bóng (bỏ qua khoảng cách)

In [17]:
def wrong_count_only(boxes):
    total = boxes.count("1")
    return [total - (c == "1") for c in boxes]

b, exp = examples[1]
print("sai :", wrong_count_only(b), "| đúng:", exp)

sai : [3, 3, 2, 3, 2, 2] | đúng: [11, 8, 5, 4, 3, 4]


**Vì sao sai:** mỗi thao tác chỉ dịch **một bước**; bóng ở xa cần nhiều thao tác. **Sửa:** chi phí là `Σ |i − j|`.

### Lỗi 4 — Cập nhật `ans` tại chỗ trong khi còn đọc trạng thái (hiểu sai "tính độc lập")

In [18]:
def wrong_mutate(boxes):
    state = [int(c) for c in boxes]; n = len(state); ans = []
    for i in range(n):
        ops = sum(abs(i - j) * state[j] for j in range(n))
        ans.append(ops)
        state = [0] * n; state[i] = sum(int(c) for c in boxes)      # SAI: tính tiếp từ trạng thái đã dồn
    return ans

b, exp = examples[0]
print("sai :", wrong_mutate(b), "| đúng:", exp)

sai : [1, 2, 2] | đúng: [1, 1, 3]


**Vì sao sai:** đề nói mỗi `answer[i]` tính từ **trạng thái ban đầu**, không phải nối tiếp. **Sửa:** không thay đổi dữ liệu gốc giữa các `i`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Với mỗi vị trí, tổng khoảng cách tới một tập điểm" ⇒ **hai lượt quét** hoặc **tổng tiền tố của vị trí**.
- Dịch đích một bước: chi phí thay đổi `+(#bên trái) − (#bên phải)`.
- Thứ tự cập nhật trong vòng lặp (cộng `ans` trước hay cập nhật `cnt` trước) quyết định đúng sai — trace tay một ví dụ nhỏ.
- `n ≤ 2000` cho phép `O(n²)`, nhưng luôn chuẩn bị `O(n)`.

## 15. Biến thể của bài

1. **Bóng có trọng số** (`boxes[i]` = số bóng): thay `+1` bằng `+boxes[i]` — Bài tập 5.
2. **Tìm hộp có chi phí nhỏ nhất**: đó là **trung vị** các vị trí bóng (**462 Minimum Moves to Equal Array Elements II**).
3. **2D**: `Σ (|x − xj| + |y − yj|)` tách hai chiều độc lập (**296 Best Meeting Point**, Premium).
4. **2615 Sum of Distances**: tổng khoảng cách tới các phần tử bằng nhau — tiền tố vị trí theo nhóm.
5. **1685 Sum of Absolute Differences in a Sorted Array**: cùng công thức `i·cL − sL + sR − i·cR`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | `Σ |i−j|` mọi cặp | `O(n²)` | `O(1)` | Kiểm chứng, `n` nhỏ |
| 2. Hai mảng | `left[i] + right[i]` | `O(n)` | `O(n)` | Dễ hiểu |
| 3. Hai lượt | Cộng thẳng vào `ans` | `O(n)` | `O(1)` | **Bản nộp** |
| 4. Công thức tiền tố | `i·cL − sL + sR − i·cR` | `O(n)` | `O(n)` | Truy vấn `O(1)` |
| 5. Cập nhật dần | `ans[i+1] = ans[i] + L − R` | `O(n)` | `O(1)` | Ngắn gọn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `"1"`; (b) `"100"`; (c) `"0110"`; (d) `"10001"`.

**Bài 2 (tính tay):** Tính `answer` cho `"1010"` bằng Cách 5 (cập nhật dần), ghi `left`, `right` ở mỗi bước.

**Bài 3 (điền TODO):** Hoàn thiện `left_costs(boxes)` trả mảng `L` (chi phí dồn bóng bên trái về mỗi `i`).

**Bài 4 (tự viết lại):** Viết `best_box(boxes)` trả `(chỉ số nhỏ nhất có chi phí nhỏ nhất, chi phí)`; chứng minh/kiểm tra nó là một trung vị của các vị trí bóng.

**Bài 5 (mini challenge):** Viết `min_ops_weighted(counts)` với `counts[i]` = số bóng ở hộp `i` (có thể > 1), `O(n)`.

In [19]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def left_costs(boxes):
    n = len(boxes); L = [0] * n; cnt = 0
    for i in range(1, n):
        # TODO: cộng bóng ở i-1 vào cnt, rồi L[i] = L[i-1] + cnt
        pass
    return L

In [20]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def best_box(boxes):
    # TODO: ans = Solution().minOperations(boxes); trả (argmin, min)
    pass

def min_ops_weighted(counts):
    # TODO: như Cách 3 nhưng cnt += counts[i]
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: liệt kê vị trí bóng. Mức 2: `Σ |i − j|`. Mức 3: (a) `[0]`; (b) `[0,1,2]`; (c) `[3,1,1,3]`; (d) `[4,4,4,4,4]`.

**Bài 2.** Mức 1: bóng ở 0, 2; `ans[0] = 0 + 2 = 2`. Mức 2: `i=0`: left=1, right=1 ⇒ `ans[1] = 2`; `i=1`: left=1, right=1 ⇒ `ans[2] = 2`. Mức 3: `i=2`: left=2, right=0 ⇒ `ans[3] = 4` ⇒ `[2,2,2,4]`.

**Bài 3.** Mức 1: `cnt += boxes[i-1] == '1'`. Mức 2: `L[i] = L[i-1] + cnt`. Mức 3: trả `L`.

**Bài 4.** Mức 1: tính `ans`. Mức 2: `min(range(n), key=ans.__getitem__)`. Mức 3: hàm `f(i) = Σ|i − j|` lồi, đạt min tại trung vị; với số bóng chẵn mọi điểm giữa hai trung vị đều tối ưu.

**Bài 5.** Mức 1: giống Cách 3. Mức 2: `cnt += counts[i]`. Mức 3: đối chiếu vét cạn `Σ counts[j]·|i − j|`.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2, 3
b1 = [Solution().minOperations(b) for b in ["1", "100", "0110", "10001"]]
print("Bài 1:", b1); assert b1 == [[0], [0, 1, 2], [3, 1, 1, 3], [4, 4, 4, 4, 4]]
print("Bài 2:", ops_shift("1010")); assert ops_shift("1010") == [2, 2, 2, 4]
def left_costs(boxes):
    n = len(boxes); L = [0] * n; cnt = 0
    for i in range(1, n):
        cnt += boxes[i - 1] == "1"
        L[i] = L[i - 1] + cnt
    return L
rng = random.Random(1)
for _ in range(300):
    b = "".join(rng.choice("01") for _ in range(rng.randint(1, 20)))
    assert left_costs(b) == [sum(i - j for j in range(i) if b[j] == "1") for i in range(len(b))]
print("Bài 3: OK", left_costs("001011"))

Bài 1: [[0], [0, 1, 2], [3, 1, 1, 3], [4, 4, 4, 4, 4]]
Bài 2: [2, 2, 2, 4]
Bài 3: OK [0, 0, 0, 1, 2, 4]


In [22]:
# Bài 4, 5
def best_box(boxes):
    ans = Solution().minOperations(boxes)
    i = min(range(len(ans)), key=ans.__getitem__)
    return i, ans[i]

def min_ops_weighted(counts):
    n = len(counts); ans = [0] * n
    cnt = cost = 0
    for i in range(n):
        ans[i] += cost; cnt += counts[i]; cost += cnt
    cnt = cost = 0
    for i in range(n - 1, -1, -1):
        ans[i] += cost; cnt += counts[i]; cost += cnt
    return ans

rng = random.Random(2)
for _ in range(300):
    b = "".join(rng.choice("01") for _ in range(rng.randint(1, 20)))
    if "1" in b:
        pos = [j for j, c in enumerate(b) if c == "1"]
        i, v = best_box(b)
        lo, hi = pos[(len(pos) - 1) // 2], pos[len(pos) // 2]          # đoạn trung vị
        assert lo <= i <= hi or Solution().minOperations(b)[lo] == v
    counts = [rng.randint(0, 5) for _ in range(rng.randint(1, 15))]
    assert min_ops_weighted(counts) == [sum(counts[j] * abs(i - j) for j in range(len(counts))) for i in range(len(counts))]
print("Bài 4, 5: OK |", best_box("001011"), min_ops_weighted([2, 0, 1]))

Bài 4, 5: OK | (4, 3) [2, 3, 4]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "tổng khoảng cách tới tập điểm, hai lượt":**

```python
ans = [0]*n; cnt = cost = 0
for i in range(n):            # trái -> phải
    ans[i] += cost; cnt += w[i]; cost += cnt
cnt = cost = 0
for i in range(n-1, -1, -1):  # phải -> trái
    ans[i] += cost; cnt += w[i]; cost += cnt
```

**Checklist:**
- [ ] Cộng `cost` vào `ans[i]` **trước** khi cập nhật `cnt`, `cost`?
- [ ] Có cả hai lượt (bóng bên trái **và** bên phải)?
- [ ] Mỗi `answer[i]` tính từ trạng thái **ban đầu**?
- [ ] Đã test `n = 1`, toàn `0`, toàn `1`, bóng ở hai đầu?